# Lab 02 - PySpark Transformation Lab (Hive Metastore)

**Durée:** ~60 minutes

Ce lab couvre les transformations PySpark fondamentales en utilisant des données générées directement dans le Hive Metastore — **aucune configuration externe requise**.

**Techniques couvertes:**
- Lecture de données (CSV, JSON)
- Filtrage avec conditions multiples
- Colonnes calculées (CASE WHEN, extraction de dates)
- Jointures
- Agrégations (groupBy)
- Fonctions de fenêtre (rank, running total, lag/lead)

---
## Step 0: Génération des données

> Exécutez cette cellule pour créer les tables dans le Hive Metastore. Cela prend ~10 secondes.

In [ ]:
# ============================================================
# DATA GENERATION — Creates tables in Hive Metastore
# Run this cell ONCE before starting the exercises
# ============================================================

from pyspark.sql.types import *
from pyspark.sql.functions import *
import random

spark.sql("CREATE DATABASE IF NOT EXISTS training_lab")
spark.sql("USE training_lab")

# --- Reinsurance Contracts (500 records) ---
entity_ids = [f"ENT_{str(i).zfill(3)}" for i in range(1, 19)]
reinsurer_ids = ["SWISS-RE-001", "MUNICH-RE-001", "HANNOVER-RE-001", "SCOR-RE-001", "LLOYDS-001"]
contract_types = ["QUOTA_SHARE", "EXCESS_OF_LOSS", "STOP_LOSS", "FACULTATIVE"]
currencies = ["EUR", "USD", "GBP", "CHF"]
statuses = ["ACTIVE", "EXPIRED", "PENDING"]

contracts_data = []
for i in range(1, 501):
    contracts_data.append((
        f"RC-{str(i).zfill(6)}",
        random.choice(entity_ids),
        random.choice(reinsurer_ids),
        random.choice(contract_types),
        f"2020-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
        f"2025-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
        round(random.uniform(100000, 50000000), 2),
        round(random.uniform(0.05, 0.95), 4),
        random.choice(currencies),
        random.choice(statuses)
    ))

contracts_schema = StructType([
    StructField("contract_id", StringType()), StructField("axa_entity_id", StringType()),
    StructField("reinsurer_id", StringType()), StructField("contract_type", StringType()),
    StructField("effective_date", StringType()), StructField("expiry_date", StringType()),
    StructField("premium_amount", DoubleType()), StructField("cession_rate", DoubleType()),
    StructField("currency", StringType()), StructField("status", StringType())
])

contracts_df_gen = spark.createDataFrame(contracts_data, contracts_schema) \
    .withColumn("effective_date", to_date(col("effective_date"))) \
    .withColumn("expiry_date", to_date(col("expiry_date")))
contracts_df_gen.write.mode("overwrite").saveAsTable("training_lab.reinsurance_contracts")

# --- AXA Entities (18 records) ---
entity_types = ["Insurance", "Reinsurance", "Life Insurance", "Asset Management"]
regions = ["Europe", "Americas", "Asia Pacific", "Global"]
countries = ["France", "Germany", "UK", "Switzerland", "USA", "Japan", "Brazil", "Singapore", "Belgium", "Italy", "Spain", "Hong Kong", "Mexico", "Australia", "Ireland", "Netherlands", "Canada", "South Korea"]

entities_data = [(f"ENT_{str(i).zfill(3)}", f"AXA {countries[i-1]}", 
                  entity_types[i % len(entity_types)], regions[i % len(regions)], countries[i-1])
                 for i in range(1, 19)]

entities_schema = StructType([
    StructField("entity_id", StringType()), StructField("entity_name", StringType()),
    StructField("entity_type", StringType()), StructField("region", StringType()),
    StructField("country", StringType())
])

entities_df_gen = spark.createDataFrame(entities_data, entities_schema)
entities_df_gen.write.mode("overwrite").saveAsTable("training_lab.axa_entities")

print("=" * 50)
print("DATA GENERATED SUCCESSFULLY")
print("=" * 50)
print(f"Contracts: {spark.table('training_lab.reinsurance_contracts').count()} rows")
print(f"Entities:  {spark.table('training_lab.axa_entities').count()} rows")
print("\nTables: training_lab.reinsurance_contracts, training_lab.axa_entities")

---
## Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, sum as spark_sum, count, avg, max as spark_max, min as spark_min,
    round as spark_round, concat, concat_ws, year, month, date_format, desc, asc,
    row_number, rank, dense_rank, lag, lead, ntile,
    explode, array, array_contains, array_distinct, collect_list
)
from pyspark.sql.window import Window

print("Libraries imported.")

---
## Section 1: Lecture des données

### Exercise 1.1 - Lire la table des contrats de réassurance

In [ ]:
# Exercise 1.1 - Read reinsurance contracts from Hive Metastore
# HINT: spark.read.table("database.table")

contracts_df = spark.read.table("training_lab.reinsurance_contracts")

print(f"Contracts: {contracts_df.count()} rows")
contracts_df.printSchema()
display(contracts_df.limit(5))

In [ ]:
# Verification 1.1
assert contracts_df.count() > 0, "DataFrame is empty"
assert "premium_amount" in contracts_df.columns
print("Exercise 1.1 passed!")

### Exercise 1.2 - Lire la table des entités AXA

In [ ]:
# Exercise 1.2 - Read AXA entities

entities_df = spark.read.table("training_lab.axa_entities")

print(f"Entities: {entities_df.count()} rows")
entities_df.printSchema()
display(entities_df)

In [ ]:
# Verification 1.2
assert entities_df.count() == 18, "Should have 18 entities"
print("Exercise 1.2 passed!")

### Enregistrer des vues temporaires pour comparaison SQL

In [ ]:
contracts_df.createOrReplaceTempView("contracts")
entities_df.createOrReplaceTempView("entities")
print("Temp views created: contracts, entities")

---
## Section 2: Filtrage

### Exercise 2.1 - Filtrer avec conditions multiples

In [ ]:
%sql
-- Équivalent SQL
SELECT * FROM contracts
WHERE status = 'ACTIVE' AND premium_amount > 5000000 AND currency = 'EUR'

In [ ]:
# Exercise 2.1 - Filter: ACTIVE contracts with premium > 5M in EUR
# TODO: Fill in the filter conditions

high_value_active = contracts_df.filter(
    (col("status") == "ACTIVE") &
    (col("premium_amount") > 5000000) &
    (col("currency") == "EUR")
)

print(f"High-value active EUR contracts: {high_value_active.count()}")
display(high_value_active.limit(10))

In [ ]:
# Verification 2.1
assert high_value_active.count() > 0, "No results found"
assert high_value_active.filter(col("status") != "ACTIVE").count() == 0
print("Exercise 2.1 passed!")

### Exercise 2.2 - Filtrer avec IN et sélection de colonnes

In [ ]:
%sql
-- Équivalent SQL
SELECT contract_id, reinsurer_id, premium_amount, status
FROM contracts
WHERE reinsurer_id IN ('SWISS-RE-001', 'MUNICH-RE-001', 'HANNOVER-RE-001')
  AND status = 'ACTIVE'
ORDER BY premium_amount DESC

In [ ]:
# Exercise 2.2 - Filter with isin + select + orderBy
# TODO: Fill in the isin values

target_reinsurers = ["SWISS-RE-001", "MUNICH-RE-001", "HANNOVER-RE-001"]

selected_reinsurers = contracts_df \
    .filter(col("reinsurer_id").isin(target_reinsurers)) \
    .filter(col("status") == "ACTIVE") \
    .select("contract_id", "reinsurer_id", "premium_amount", "status") \
    .orderBy(col("premium_amount").desc())

print(f"Selected: {selected_reinsurers.count()} contracts")
display(selected_reinsurers.limit(10))

In [ ]:
# Verification 2.2
assert selected_reinsurers.count() > 0
print("Exercise 2.2 passed!")

---
## Section 3: Colonnes calculées

### Exercise 3.1 - Colonne conditionnelle (CASE WHEN)

In [ ]:
%sql
-- Équivalent SQL
SELECT *, CASE 
    WHEN premium_amount > 10000000 THEN 'HIGH'
    WHEN premium_amount > 1000000  THEN 'MEDIUM'
    ELSE 'LOW' 
  END AS premium_category
FROM contracts

In [ ]:
# Exercise 3.1 - Add premium_category using when/otherwise
# HINT: when(cond, val).when(cond, val).otherwise(val)

contracts_categorized = contracts_df.withColumn(
    "premium_category",
    when(col("premium_amount") > 10000000, "HIGH")
    .when(col("premium_amount") > 1000000, "MEDIUM")
    .otherwise("LOW")
)

display(contracts_categorized.select("contract_id", "premium_amount", "premium_category").limit(10))

In [ ]:
# Verification 3.1
assert "premium_category" in contracts_categorized.columns
print("Exercise 3.1 passed!")

### Exercise 3.2 - Extraction de dates

In [ ]:
%sql
-- Équivalent SQL
SELECT contract_id, effective_date, 
       YEAR(effective_date) AS effective_year,
       MONTH(effective_date) AS effective_month
FROM contracts

In [ ]:
# Exercise 3.2 - Extract year and month from effective_date
# HINT: year(col), month(col)

contracts_with_year = contracts_df \
    .withColumn("effective_year", year(col("effective_date"))) \
    .withColumn("effective_month", month(col("effective_date")))

display(contracts_with_year.select("contract_id", "effective_date", "effective_year", "effective_month").limit(10))

In [ ]:
# Verification 3.2
assert "effective_year" in contracts_with_year.columns
assert "effective_month" in contracts_with_year.columns
print("Exercise 3.2 passed!")

---
## Section 4: Jointures

### Exercise 4.1 - Inner Join (Contracts ↔ Entities)

In [ ]:
%sql
-- Équivalent SQL
SELECT c.*, e.entity_name, e.region, e.country
FROM contracts c
INNER JOIN entities e ON c.axa_entity_id = e.entity_id

In [ ]:
# Exercise 4.1 - Inner join contracts with entities
# TODO: Fill in join column and join type

joined_df = contracts_df.join(
    entities_df,
    contracts_df["axa_entity_id"] == entities_df["entity_id"],
    "inner"
)

print(f"Joined rows: {joined_df.count()}")
display(joined_df.select("contract_id", "entity_name", "region", "premium_amount").limit(10))

In [ ]:
# Verification 4.1
assert joined_df.count() > 0, "Join produced no results"
assert "entity_name" in joined_df.columns
print("Exercise 4.1 passed!")

### Exercise 4.2 - Sélectionner des colonnes après jointure

In [ ]:
# Exercise 4.2 - Select specific columns from joined result
# Avoid ambiguous column names by selecting explicitly

enriched_contracts = joined_df.select(
    col("contract_id"),
    col("entity_name"),
    col("region"),
    col("country"),
    col("contract_type"),
    col("premium_amount"),
    col("cession_rate"),
    col("currency"),
    col("status")
)

display(enriched_contracts.limit(10))

---
## Section 5: Agrégations

### Exercise 5.1 - Agrégation par région

In [ ]:
%sql
-- Équivalent SQL
SELECT region, 
       COUNT(*) AS contract_count,
       ROUND(SUM(premium_amount), 2) AS total_premium,
       ROUND(AVG(premium_amount), 2) AS avg_premium
FROM contracts c JOIN entities e ON c.axa_entity_id = e.entity_id
GROUP BY region
ORDER BY total_premium DESC

In [ ]:
# Exercise 5.1 - Aggregate by region
# TODO: Fill in the aggregation function

region_summary = enriched_contracts.groupBy("region").agg(
    count("*").alias("contract_count"),
    spark_round(spark_sum("premium_amount"), 2).alias("total_premium"),
    spark_round(avg("premium_amount"), 2).alias("avg_premium")
).orderBy(col("total_premium").desc())

display(region_summary)

In [ ]:
# Verification 5.1
assert region_summary.count() > 0
assert "contract_count" in region_summary.columns
print("Exercise 5.1 passed!")

### Exercise 5.2 - Agrégation par région ET type de contrat

In [ ]:
%sql
-- Équivalent SQL
SELECT region, contract_type, COUNT(*) AS cnt, ROUND(AVG(premium_amount),2) AS avg_prem
FROM contracts c JOIN entities e ON c.axa_entity_id = e.entity_id
GROUP BY region, contract_type
ORDER BY region, avg_prem DESC

In [ ]:
# Exercise 5.2 - Aggregate by region AND contract_type

region_type_summary = enriched_contracts.groupBy("region", "contract_type").agg(
    count("*").alias("contract_count"),
    spark_round(avg("premium_amount"), 2).alias("avg_premium")
).orderBy("region", col("avg_premium").desc())

display(region_type_summary)

In [ ]:
# Verification 5.2
assert region_type_summary.count() > region_summary.count()
print("Exercise 5.2 passed!")

---
## Section 6: Fonctions de fenêtre

### Exercise 6.1 - Classement par prime au sein de chaque entité (dense_rank)

In [ ]:
%sql
-- Équivalent SQL
SELECT contract_id, axa_entity_id, premium_amount,
       DENSE_RANK() OVER (PARTITION BY axa_entity_id ORDER BY premium_amount DESC) AS premium_rank
FROM contracts
QUALIFY premium_rank <= 3

In [ ]:
# Exercise 6.1 - Rank contracts by premium within each entity
# TODO: Fill in partitionBy and the ranking function

entity_premium_window = Window.partitionBy("axa_entity_id").orderBy(desc("premium_amount"))

ranked_contracts = contracts_df \
    .withColumn("premium_rank", dense_rank().over(entity_premium_window)) \
    .filter(col("premium_rank") <= 3) \
    .select("contract_id", "axa_entity_id", "premium_amount", "premium_rank")

display(ranked_contracts.orderBy("axa_entity_id", "premium_rank").limit(15))

In [ ]:
# Verification 6.1
assert "premium_rank" in ranked_contracts.columns
assert ranked_contracts.filter(col("premium_rank") > 3).count() == 0
print("Exercise 6.1 passed!")

### Exercise 6.2 - Total cumulé des primes par entité

In [ ]:
# Exercise 6.2 - Running total of premiums by entity, ordered by effective_date
# HINT: Window with rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_window = Window.partitionBy("axa_entity_id").orderBy("effective_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

contracts_running = contracts_df \
    .withColumn("running_premium_total", spark_round(spark_sum("premium_amount").over(running_window), 2)) \
    .select("axa_entity_id", "effective_date", "premium_amount", "running_premium_total")

display(contracts_running.filter(col("axa_entity_id") == "ENT_001").orderBy("effective_date").limit(10))

In [ ]:
# Verification 6.2
assert "running_premium_total" in contracts_running.columns
print("Exercise 6.2 passed!")

### Exercise 6.3 - Comparer avec le contrat précédent (Lag / Lead)

In [ ]:
# Exercise 6.3 - Compare each contract with previous and next premium
# HINT: lag("column", 1) for previous, lead("column", 1) for next

seq_window = Window.partitionBy("axa_entity_id").orderBy("effective_date")

contracts_compare = contracts_df \
    .withColumn("prev_premium", lag("premium_amount", 1).over(seq_window)) \
    .withColumn("next_premium", lead("premium_amount", 1).over(seq_window)) \
    .withColumn("premium_change", spark_round(col("premium_amount") - col("prev_premium"), 2)) \
    .select("axa_entity_id", "effective_date", "premium_amount", "prev_premium", "next_premium", "premium_change")

display(contracts_compare.filter(col("axa_entity_id") == "ENT_001").orderBy("effective_date").limit(10))

In [ ]:
# Verification 6.3
assert "prev_premium" in contracts_compare.columns
assert "next_premium" in contracts_compare.columns
print("Exercise 6.3 passed!")

---
## Section 7: Challenge

> Combinez toutes les techniques dans un pipeline multi-étapes.

**Objectif:** Pour chaque région, trouvez les 3 entités avec le plus grand volume total de primes, avec leur rang et le % de la région.

In [ ]:
# Challenge: Multi-step pipeline
# 1. Join contracts with entities
# 2. Aggregate total premium per entity per region
# 3. Rank entities within each region
# 4. Filter top 3 per region
# 5. Add percentage of region total

# Step 1 + 2
entity_totals = contracts_df.join(
    entities_df,
    contracts_df["axa_entity_id"] == entities_df["entity_id"],
    "inner"
).groupBy("entity_id", "entity_name", "region").agg(
    spark_round(spark_sum("premium_amount"), 2).alias("total_premium"),
    count("contract_id").alias("contract_count")
)

# Step 3
w = Window.partitionBy("region").orderBy(col("total_premium").desc())
ranked = entity_totals.withColumn("region_rank", dense_rank().over(w))

# Step 4
top3 = ranked.filter(col("region_rank") <= 3)

# Step 5
region_totals = entity_totals.groupBy("region").agg(spark_sum("total_premium").alias("region_total"))
result = top3.join(region_totals, "region") \
    .withColumn("pct_of_region", spark_round(col("total_premium") / col("region_total") * 100, 2)) \
    .select("region", "region_rank", "entity_name", "total_premium", "contract_count", "pct_of_region") \
    .orderBy("region", "region_rank")

display(result)

In [ ]:
# Verification Challenge
assert "pct_of_region" in result.columns
assert result.filter(col("region_rank") > 3).count() == 0
print("Challenge passed! Bravo!")

---
## Nettoyage (optionnel)

Décommentez pour supprimer les tables créées.

In [ ]:
# Uncomment to clean up:
# spark.sql("DROP TABLE IF EXISTS training_lab.reinsurance_contracts")
# spark.sql("DROP TABLE IF EXISTS training_lab.axa_entities")
# spark.sql("DROP DATABASE IF EXISTS training_lab CASCADE")
# print("Cleaned up.")

---
## Lab terminé !

Techniques maîtrisées :
- ✅ Lecture depuis Hive Metastore
- ✅ Filtrage avec conditions multiples et IN
- ✅ Colonnes calculées (CASE WHEN, dates)
- ✅ Jointures
- ✅ Agrégations (groupBy, count, sum, avg)
- ✅ Fonctions de fenêtre (dense_rank, running total, lag/lead)
- ✅ Pipeline multi-étapes